# 03_gradcam.ipynb

Generate Grad-CAM overlays for fraud predictions and visualize the model's salient image regions.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
sys.path.append(str(Path('..').resolve()))
from src.model import FraudCNN
from src.gradcam_utils import compute_gradcam, overlay_gradcam
from src.dataset import TransactionImageDataset
from src.train import load_checkpoint

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = FraudCNN(input_channels=1, num_classes=2)
model = load_checkpoint(model, str(Path('..') / 'outputs' / 'models' / 'fraud_cnn_gaf.pth'), device)

In [ ]:
dataset = np.load(Path('..') / 'data' / 'transactions_gaf_28.npz')
images = dataset['images']
labels = dataset['labels']
fraud_indices = np.where(labels == 1)[0]
sample_idx = fraud_indices[:10] if len(fraud_indices) >= 10 else fraud_indices
image_batch = torch.from_numpy(images[sample_idx]).unsqueeze(1)

target_layer = model.features[-1]
cam_maps = []
overlays = []
for img_tensor in image_batch:
    grayscale_cam = compute_gradcam(model, img_tensor.unsqueeze(0), target_layer, target_category=1, use_cuda=torch.cuda.is_available())
    image_np = img_tensor.squeeze().numpy()
    overlays.append(overlay_gradcam(image_np, grayscale_cam))
    cam_maps.append(grayscale_cam)

fig, axes = plt.subplots(2, len(overlays), figsize=(20, 6))
for i, overlay in enumerate(overlays):
    axes[0, i].imshow(images[sample_idx[i]], cmap='viridis')
    axes[0, i].axis('off')
    axes[0, i].set_title(f'Fraud {i+1}')
    axes[1, i].imshow(overlay)
    axes[1, i].axis('off')
    axes[1, i].set_title('Grad-CAM')
plt.tight_layout()

## Interpretation

The top row above shows encoded fraud images and the bottom row highlights the image regions that most strongly activated the fraud class. Analysts can use these overlays to compare visual signatures between fraud and legitimate transactions.